In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
import os
import sys

print("Current directory:", os.getcwd())
print("Python version:", sys.version)

print("Repository files:")
print(os.listdir("."))

In [0]:
from pathlib import Path
import sys

# Current notebook directory: <repo>/notebooks
notebook_dir = Path.cwd()

# Repository root: <repo>
repo_root = notebook_dir.parent

# Python source directory: <repo>/src
src_path = repo_root / "src"

print("Notebook directory:", notebook_dir)
print("Repository root:", repo_root)
print("Source directory:", src_path)

# Verify the expected project structure
assert (repo_root / "pyproject.toml").exists() or (
    repo_root / "setup.py"
).exists() or (repo_root / "src" / "ssis_migration").is_dir(), (
    "Could not verify the repository structure."
)

assert (src_path / "ssis_migration").is_dir(), (
    f"Migration package not found at {src_path / 'ssis_migration'}"
)

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import ssis_migration

print("\nMigration engine import: SUCCESS")
print("Package location:", ssis_migration.__file__)

In [0]:
from pathlib import Path
import importlib
import sys

repo_root = Path.cwd().parent
src_path = repo_root / "src"

print("=== PROJECT CONFIGURATION ===")

for name in (
    "pyproject.toml",
    "requirements.txt",
    "setup.py",
    "setup.cfg",
    "pytest.ini",
):
    path = repo_root / name
    if path.is_file():
        print(f"\n--- {name} ---")
        print(path.read_text()[:8000])

print("\n=== SSIS SAMPLE PACKAGES ===")

packages = list(repo_root.rglob("*.dtsx"))

for package in packages[:30]:
    print(package.relative_to(repo_root))

print(f"\nTotal DTSX packages found: {len(packages)}")

print("\n=== CORE MODULE IMPORTS ===")

modules = [
    "ssis_migration.parser.package_parser",
    "ssis_migration.ir.builder",
    "ssis_migration.analyzer.package",
    "ssis_migration.planner.planner",
]

for module_name in modules:
    try:
        importlib.import_module(module_name)
        print(f"OK      {module_name}")
    except Exception as exc:
        print(f"FAILED  {module_name}: {type(exc).__name__}: {exc}")

In [0]:
from pathlib import Path
import inspect

repo_root = Path.cwd().parent
transformation_path = (
    repo_root / "src" / "ssis_migration"
    / "analyzer" / "transformation.py"
)

print(transformation_path.read_text())

In [0]:
from pathlib import Path
import ssis_migration.planner as planner

planner_path = Path(planner.__file__).parent

print("PLANNER DIRECTORY:", planner_path)
print("\nFILES:")

for path in sorted(planner_path.rglob("*.py")):
    print(f"\n--- {path.relative_to(planner_path)} ---")
    print(path.read_text()[:12000])

In [0]:
import inspect

from ssis_migration.context.migration_context import MigrationContext
from ssis_migration.planner.planner import MigrationPlanner

print("MigrationContext signature:")
print(inspect.signature(MigrationContext))

print("\nMigrationContext source:")
print(inspect.getsource(MigrationContext))

print("\nMigrationPlanner.build_plan:")
print(inspect.getsource(MigrationPlanner.build_plan))

In [0]:
from pathlib import Path
import ssis_migration.context.migration_context as context_module

context_path = Path(context_module.__file__).parent

for path in sorted(context_path.glob("*.py")):
    print(f"\n--- {path.name} ---")
    print(path.read_text()[:10000])

In [0]:
from pathlib import Path

search_roots = [
    repo_root / "examples",
    repo_root / "config",
    repo_root / "scripts",
]

for root in search_roots:
    if not root.exists():
        continue

    for path in sorted(root.rglob("*")):
        if path.is_file() and path.suffix.lower() in {
            ".json", ".yaml", ".yml", ".toml"
        }:
            print(path.relative_to(repo_root))

In [0]:
from pathlib import Path
import sys
import json

repo_root = Path.cwd().parent
src_path = repo_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from ssis_migration.parser.package_parser import SSISPackageParser
from ssis_migration.ir.builder import IRBuilder
from ssis_migration.analyzer.package import analyze_package

# 1. Locate the existing SSIS package
package_path = repo_root / "examples" / "wwi" / "DailyETLMain.dtsx"
assert package_path.is_file(), f"Package not found: {package_path}"

print("Package:", package_path.name)

# 2. Parse the DTSX XML
parsed = SSISPackageParser(str(package_path)).parse()
print("STEP 1: SSIS parsing completed")

# 3. Build the intermediate representation
package_ir = IRBuilder().build(parsed)
print("STEP 2: IR construction completed")
print("IR type:", type(package_ir).__name__)

# 4. Analyze the dictionary representation of the IR
analysis = analyze_package(package_ir.to_dict())
print("STEP 3: Package analysis completed")
print("Analysis type:", type(analysis).__name__)

# 5. Inspect the actual result structure
if isinstance(analysis, dict):
    print("\nAnalysis keys:")
    print(json.dumps(list(analysis.keys()), indent=2))

    for key, value in analysis.items():
        if isinstance(value, (list, dict, str, int, float, bool, type(None))):
            print(f"{key}: {type(value).__name__}", end="")
            if isinstance(value, (list, dict)):
                print(f", count={len(value)}")
            else:
                print()
else:
    print("\nAnalysis attributes:")
    print([
        name for name in dir(analysis)
        if not name.startswith("_")
    ])

In [0]:
import inspect

from ssis_migration.database.connection_mapping import ConnectionMappingSet

print("CLASS SOURCE:")
print(inspect.getsource(ConnectionMappingSet))

print("\nCONSTRUCTOR SIGNATURE:")
print(inspect.signature(ConnectionMappingSet))

print("\nAVAILABLE METHODS:")
print([
    name for name in dir(ConnectionMappingSet)
    if not name.startswith("_")
])

In [0]:
import json
import inspect
from pathlib import Path

from ssis_migration.database.connection_mapping import (
    ConnectionMappingSet,
    ConnectionDatabaseMapping,
)

repo_root = Path.cwd().parent
wwi_dir = repo_root / "examples" / "wwi"

mapping_data = json.loads(
    (wwi_dir / "connection_mapping.json").read_text()
)

print("Mapping entries:", len(mapping_data["mappings"]))

print("ConnectionDatabaseMapping constructor:")
print(inspect.signature(ConnectionDatabaseMapping))

print("\nFirst mapping entry:")
print(json.dumps(mapping_data["mappings"][0], indent=2))

In [0]:
import json
from ssis_migration.database.connection_mapping import (
    ConnectionMappingSet,
    ConnectionDatabaseMapping,
)

mapping_data = json.loads(
    (wwi_dir / "connection_mapping.json").read_text()
)

connection_mapping = ConnectionMappingSet(
    mappings=[
        ConnectionDatabaseMapping(**item)
        for item in mapping_data["mappings"]
    ]
)

print("Connection mapping constructed successfully")
print("Mappings:", len(connection_mapping.mappings))

for mapping in connection_mapping.mappings:
    print(
        mapping.connection_reference,
        "->", mapping.database_name,
        "| role:", mapping.role,
        "| metadata:", mapping.metadata_file,
    )

# Build the migration context using existing local metadata.
context = build_migration_context(
    package_ir=package_ir,
    ssis_analysis=analysis,
    connection_mapping=connection_mapping,
    base_path=repo_root,
)

print("Migration context built")
print("Database contexts:", len(context.databases))

for db in context.databases:
    print(
        "Database:", db.database_name,
        "| Role:", db.role,
        "| Relevant objects:", len(db.relevant_objects),
        "| Dependency edges:", len(db.dependency_edges),
    )

In [0]:
from ssis_migration.planner.planner import MigrationPlanner

migration_plan = MigrationPlanner().build_plan(context)
plan_dict = migration_plan.to_dict()

print("=" * 60)
print("MIGRATION PLAN SUMMARY")
print("=" * 60)

print("Package:", plan_dict.get("package_name"))
print("Planned steps:", len(plan_dict.get("steps", [])))
print("Unresolved items:", len(plan_dict.get("unresolved", [])))
print("Plan metadata:", plan_dict.get("metadata"))

print("\n" + "=" * 60)
print("FIRST THREE LOADS")
print("=" * 60)

for step in plan_dict.get("steps", [])[:3]:
    metadata = step.get("metadata", {})

    print("\nLoad:", step.get("name"))
    print("  Source objects:", metadata.get("source_objects", []))
    print("  Target objects:", metadata.get("target_objects", []))
    print("  Source procedure semantics:",
          len(metadata.get("procedure_semantics", [])))
    print("  Destination procedure semantics:",
          len(metadata.get("destination_procedure_semantics", [])))
    print("  Migration strategy:",
          metadata.get("migration_strategy"))

print("\n" + "=" * 60)
print("UNRESOLVED ITEMS")
print("=" * 60)

for item in plan_dict.get("unresolved", [])[:10]:
    print(item)

In [0]:
import pkgutil
import inspect
import ssis_migration.generator as generator_package

print("GENERATOR PACKAGE:", generator_package.__file__)

for module_info in pkgutil.iter_modules(
    generator_package.__path__
):
    module_name = module_info.name
    module = __import__(
        f"ssis_migration.generator.{module_name}",
        fromlist=["*"],
    )

    print(f"\nMODULE: {module_name}")

    for name, obj in inspect.getmembers(module):
        if name.startswith("_"):
            continue

        if inspect.isclass(obj) or inspect.isfunction(obj):
            if getattr(obj, "__module__", None) == module.__name__:
                try:
                    signature = inspect.signature(obj)
                except (TypeError, ValueError):
                    signature = "(signature unavailable)"

                print(f"  {name}{signature}")

In [0]:
import inspect

from ssis_migration.generator.dispatcher import GeneratorDispatcher
from ssis_migration.generator.base import MigrationGenerator

print("=" * 60)
print("GENERATOR DISPATCHER")
print("=" * 60)
print(inspect.getsource(GeneratorDispatcher))

print("\n" + "=" * 60)
print("BASE GENERATOR")
print("=" * 60)
print(inspect.getsource(MigrationGenerator))

In [0]:
from ssis_migration.generator.dispatcher import GeneratorDispatcher

steps = plan_dict.get("steps", [])
assert steps, "Migration plan contains no steps"

print("First step keys:", list(steps[0].keys()))

dispatcher = GeneratorDispatcher()

artifact = dispatcher.generate(steps[0])

print("\n" + "=" * 60)
print("GENERATED ARTIFACT")
print("=" * 60)

print("Artifact ID:", artifact.artifact_id)
print("Name:", artifact.name)
print("Type:", artifact.artifact_type)
print("Target platform:", artifact.target_platform)
print("Language:", artifact.language)
print("Source step ID:", artifact.source_step_id)
print("Code length:", len(artifact.code))
print("Artifact metadata keys:", list(artifact.metadata.keys()))

print("\n" + "=" * 60)
print("GENERATED CODE PREVIEW")
print("=" * 60)
print(artifact.code[:5000])

In [0]:
import pkgutil
import inspect
import ssis_migration.agent as agent_package

print("AGENT PACKAGE:", agent_package.__file__)

for item in pkgutil.iter_modules(agent_package.__path__):
    module_name = item.name

    try:
        module = __import__(
            f"ssis_migration.agent.{module_name}",
            fromlist=["*"],
        )
    except Exception as exc:
        print(f"\nMODULE: {module_name} — import failed: {exc}")
        continue

    print(f"\nMODULE: {module_name}")

    for name, obj in inspect.getmembers(module):
        if name.startswith("_"):
            continue

        if (
            inspect.isclass(obj) or inspect.isfunction(obj)
        ) and getattr(obj, "__module__", None) == module.__name__:
            try:
                signature = inspect.signature(obj)
            except (TypeError, ValueError):
                signature = "(signature unavailable)"

            print(f"  {name}{signature}")

In [0]:
from pathlib import Path
import sys
import inspect
import pkgutil

# Restore the repository import path for this notebook cell.
repo_root = Path.cwd().parent
src_path = repo_root / "src"

if not src_path.is_dir():
    raise FileNotFoundError(f"Source directory not found: {src_path}")

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import ssis_migration.agent as agent_package

from ssis_migration.agent.orchestrator import AIMigrationOrchestrator
from ssis_migration.agent.context_builder import AIMigrationContextBuilder
from ssis_migration.agent.prompt_builder import AIMigrationPromptBuilder
from ssis_migration.agent.provider_factory import create_provider
from ssis_migration.agent.response_validator import AIResponseValidator

print("Repository:", repo_root)
print("Source path:", src_path)
print("Agent package:", agent_package.__file__)

classes = [
    AIMigrationOrchestrator,
    AIMigrationContextBuilder,
    AIMigrationPromptBuilder,
    AIResponseValidator,
]

for cls in classes:
    print("\n" + "=" * 75)
    print(f"{cls.__module__}.{cls.__name__}")
    print("=" * 75)

    for name, method in inspect.getmembers(cls, predicate=inspect.isfunction):
        if name.startswith("__"):
            continue

        try:
            signature = inspect.signature(method)
            source = inspect.getsource(method)
        except (TypeError, OSError) as exc:
            print(f"\n{name}: source unavailable ({exc})")
            continue

        print(f"\n--- {name}{signature} ---")
        print(source[:7000])

print("\n" + "=" * 75)
print("PROVIDER FACTORY")
print("=" * 75)
print(inspect.getsource(create_provider))

In [0]:
from pathlib import Path
import sys
import inspect

repo_root = Path.cwd().parent
src_path = repo_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from ssis_migration.agent.planner import AIMigrationPlanner
from ssis_migration.agent.provider import AIProvider
from ssis_migration.agent.openai_provider import OpenAIProvider
from ssis_migration.agent.puter_provider import PuterAIProvider
from ssis_migration.agent.orchestrator import AIMigrationOrchestrator

print("REQUEST PLANNER")
print(inspect.getsource(AIMigrationPlanner))

print("\nORCHESTRATOR CONSTRUCTOR")
print(inspect.signature(AIMigrationOrchestrator))
print(inspect.getsource(AIMigrationOrchestrator.__init__))

print("\nPROVIDER INTERFACE")
print(inspect.getsource(AIProvider))

print("\nOPENAI PROVIDER CONSTRUCTOR")
print(inspect.signature(OpenAIProvider))

print("\nPUTER PROVIDER CONSTRUCTOR")
print(inspect.signature(PuterAIProvider))

In [0]:
import os
import importlib.util

print("Anthropic SDK installed:",
      importlib.util.find_spec("anthropic") is not None)

print("ANTHROPIC_API_KEY configured:",
      bool(os.environ.get("ANTHROPIC_API_KEY")))

print("Repository root:", repo_root)
print("Agent directory:",
      (repo_root / "src/ssis_migration/agent").exists())

Install Claude

In [0]:
import anthropic
import os

print("Anthropic SDK version:", anthropic.__version__)
os.environ["ANTHROPIC_API_KEY"] = (
    dbutils.secrets.get(
        scope="<your-secret-scope>",
        key="<your-secret-key>"
    )
)

print("Claude API key configured:", bool(os.environ.get("ANTHROPIC_API_KEY")))

In [0]:
scopes = dbutils.secrets.listScopes()

print("Available secret scopes:")
for scope in scopes:
    print("-", scope.name)

In [0]:
import os
import sys
import shutil
import platform
import subprocess
from pathlib import Path

print("Python platform:", platform.platform())
print("Machine architecture:", platform.machine())
print("Python executable:", sys.executable)

node_path = shutil.which("node")
npm_path = shutil.which("npm")

print("Node path:", node_path)
print("NPM path:", npm_path)

if node_path:
    p = Path(node_path)
    print("Node is symlink:", p.is_symlink())
    print("Node target:", p.resolve())
    print("Node file size:", p.stat().st_size)

    result = subprocess.run(
        ["file", str(p.resolve())],
        capture_output=True,
        text=True,
    )
    print("Node file type:", result.stdout.strip() or result.stderr.strip())

    result = subprocess.run(
        ["ls", "-l", node_path],
        capture_output=True,
        text=True,
    )
    print("Node listing:", result.stdout.strip() or result.stderr.strip())

In [0]:
import wcwidth
from openai import OpenAI

print("wcwidth:", wcwidth.__version__)
print("OpenAI client imported successfully")

In [0]:
from openai import OpenAI

# Get the workspace URL and token from the notebook context.
workspace_url = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiUrl()
    .get()
)

token = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

client = OpenAI(
    api_key=token,
    base_url=f"{workspace_url}/ai-gateway/mlflow/v1",
)

response = client.chat.completions.create(
    model="system.ai.gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "In two sentences, what is an LLM agent?",
        }
    ],
    max_tokens=256,
)

print(response.choices[0].message.content)

In [0]:
from pathlib import Path

repo_root = Path.cwd().parent
source_dir = repo_root / "src" / "ssis_migration" / "agent"

files_to_inspect = [
    source_dir / "provider.py",
    source_dir / "openai_provider.py",
]

for file_path in files_to_inspect:
    print("\n" + "=" * 80)
    print(f"FILE: {file_path.relative_to(repo_root)}")
    print("=" * 80)

    if not file_path.exists():
        print("FILE NOT FOUND")
        continue

    print(file_path.read_text(encoding="utf-8"))

In [0]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
src_path = repo_root / "src"

print("Notebook directory:", Path.cwd())
print("Repository root:", repo_root)
print("Source directory exists:", src_path.exists())

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("Source path added:", src_path)

In [0]:
from openai import OpenAI

workspace_url = (
    "https://" + spark.conf.get("spark.databricks.workspaceUrl")
)

token = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

client = OpenAI(
    api_key=token,
    base_url=f"{workspace_url}/ai-gateway/mlflow/v1",
)

print("Databricks client initialized.")

In [0]:
from ssis_migration.agent.databricks_provider import DatabricksProvider

provider = DatabricksProvider(
    client=client,
    model="system.ai.gpt-oss-120b",
)

print("DatabricksProvider initialized successfully.")

In [0]:
test_prompt = """
Return a JSON object with exactly these fields:
- status: "REVIEW_REQUIRED"
- language: "python"
- code: "print('hello')"
- assumptions: []
- unresolved_items: []
- validation_requirements: ["Run a syntax check"]
- metadata: {}

Return only the JSON object.
"""

result = provider.generate(
    request={},
    prompt=test_prompt,
)

print("Status:", result.get("status"))
print("Language:", result.get("language"))
print("Code:", result.get("code"))
print("Provider:", result.get("metadata", {}).get("provider"))
print("Result keys:", sorted(result.keys()))

In [0]:
from pathlib import Path

factory_path = (
    Path.cwd().parent
    / "src"
    / "ssis_migration"
    / "agent"
    / "provider_factory.py"
)

print(factory_path.read_text(encoding="utf-8"))

In [0]:
import os

workspace_url = (
    "https://" + spark.conf.get("spark.databricks.workspaceUrl")
).rstrip("/")

token = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

os.environ["DATABRICKS_TOKEN"] = token
os.environ["DATABRICKS_BASE_URL"] = (
    f"{workspace_url}/ai-gateway/mlflow/v1"
)

print("Databricks provider configuration ready.")

In [0]:
from ssis_migration.agent.provider_factory import create_provider
from ssis_migration.agent.databricks_provider import DatabricksProvider

factory_provider = create_provider("databricks")

assert isinstance(factory_provider, DatabricksProvider)

print("Factory test passed:", type(factory_provider).__name__)
print("Model:", factory_provider.model)

In [0]:
import json
import sys

# Ensure the repository source directory is importable
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from ssis_migration.parser.package_parser import SSISPackageParser
from ssis_migration.ir.builder import IRBuilder
from ssis_migration.analyzer import analyze_package
from ssis_migration.database.connection_mapping import (
    ConnectionDatabaseMapping,
    ConnectionMappingSet,
)
from ssis_migration.context.builder import build_migration_context
from ssis_migration.planner.planner import MigrationPlanner

# 1. Parse the SSIS package and build its IR
package_path = repo_root / "examples" / "wwi" / "DailyETLMain.dtsx"
parsed = SSISPackageParser(str(package_path)).parse()
package_ir = IRBuilder().build(parsed)

# 2. Analyze the IR dictionary
analysis = analyze_package(package_ir.to_dict())

# 3. Load the existing connection mapping
wwi_dir = repo_root / "examples" / "wwi"
mapping_data = json.loads(
    (wwi_dir / "connection_mapping.json").read_text(encoding="utf-8")
)
connection_mapping = ConnectionMappingSet(
    mappings=[
        ConnectionDatabaseMapping(**item)
        for item in mapping_data["mappings"]
    ]
)

# 4. Build migration context and plan
context = build_migration_context(
    package_ir=package_ir,
    ssis_analysis=analysis,
    connection_mapping=connection_mapping,
    base_path=repo_root,
)
migration_plan = MigrationPlanner().build_plan(context)

# 5. Confirm the rebuilt state
print("Package:", package_ir.package_name)
print("Executables:", len(package_ir.executables))
print("Data flows:", len(package_ir.data_flows))
print("Database contexts:", len(context.databases))
print("Migration steps:", len(migration_plan.steps))

print("\nFirst migration steps:")
for step in migration_plan.steps[:5]:
    print("-", step.name)

In [0]:
from pathlib import Path

files_to_inspect = [
    repo_root / "src/ssis_migration/agent/orchestrator.py",
    repo_root / "src/ssis_migration/agent/context_builder.py",
    repo_root / "src/ssis_migration/agent/models.py",
]

for path in files_to_inspect:
    print(f"\n{'=' * 80}\n{path.relative_to(repo_root)}\n{'=' * 80}")
    print(path.read_text(encoding="utf-8"))

In [0]:
from pathlib import Path

agent_dir = repo_root / "src" / "ssis_migration" / "agent"

for path in sorted(agent_dir.glob("*.py")):
    text = path.read_text(encoding="utf-8")
    if "def create_provider" in text or "create_provider" in text:
        print(f"\n{path.name}")
        for line_no, line in enumerate(text.splitlines(), 1):
            if "create_provider" in line or "DatabricksProvider" in line:
                print(f"  {line_no}: {line.strip()}")

In [0]:
from dataclasses import asdict, is_dataclass

step = migration_plan.steps[0]

if hasattr(step, "to_dict"):
    step_data = step.to_dict()
elif is_dataclass(step):
    step_data = asdict(step)
else:
    raise TypeError(f"Unsupported step type: {type(step).__name__}")

from ssis_migration.agent.context_builder import AIMigrationContextBuilder
from ssis_migration.agent.planner import AIMigrationPlanner

ai_context = AIMigrationContextBuilder().build(
    step_data,
    package_ir=package_ir,
    databases=context.databases,
)

request = AIMigrationPlanner().build_request(ai_context)

print("Step:", step_data.get("name"))
print("Target language:", repr(request.get("task", {}).get("target_language")))
print("Task:", request.get("task"))

In [0]:
import inspect

from ssis_migration.agent.context_builder import AIMigrationContextBuilder
from ssis_migration.agent.planner import AIMigrationPlanner

print("Loaded context builder file:")
print(inspect.getfile(AIMigrationContextBuilder))

print("\nLoaded target-language logic:")
source = inspect.getsource(AIMigrationContextBuilder.build)
for line in source.splitlines():
    if "target_language" in line or "target_platform" in line:
        print(line)

step = migration_plan.steps[0]
step_data = step.to_dict()

builder = AIMigrationContextBuilder()
ai_context = builder.build(
    step_data,
    package_ir=package_ir,
    databases=context.databases,
)

print("\nContext migration settings:")
print(ai_context["migration"])

request = AIMigrationPlanner().build_request(ai_context)

print("\nFinal AI task:")
print(request["task"])

In [0]:
import importlib

import ssis_migration.agent.context_builder as context_builder_module
import ssis_migration.agent.planner as planner_module

# Reload the edited modules
importlib.reload(context_builder_module)
importlib.reload(planner_module)

# Use the newly loaded classes, not earlier imported references
builder = context_builder_module.AIMigrationContextBuilder
planner_class = planner_module.AIMigrationPlanner

step = migration_plan.steps[0]
from dataclasses import asdict

step_data = asdict(step)

ai_context = builder().build(
    step_data,
    package_ir=package_ir,
    databases=context.databases,
)

print("Context target platform:",
      repr(ai_context["migration"]["target_platform"]))
print("Context target language:",
      repr(ai_context["migration"]["target_language"]))

request = planner_class().build_request(ai_context)
print("Request task:", request["task"])

In [0]:
import importlib
import os

import ssis_migration.agent.context_builder as context_builder_module
import ssis_migration.agent.planner as planner_module
import ssis_migration.agent.orchestrator as orchestrator_module
import ssis_migration.agent.provider_factory as provider_factory_module

# Refresh edited/import-dependent modules
importlib.reload(context_builder_module)
importlib.reload(planner_module)
importlib.reload(orchestrator_module)
importlib.reload(provider_factory_module)

# Configure Databricks model access for this call
workspace_url = (
    "https://" + spark.conf.get("spark.databricks.workspaceUrl")
).rstrip("/")
token = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

os.environ["DATABRICKS_TOKEN"] = token
os.environ["DATABRICKS_BASE_URL"] = (
    f"{workspace_url}/ai-gateway/mlflow/v1"
)

try:
    provider = provider_factory_module.create_provider("databricks")
    orchestrator = orchestrator_module.AIMigrationOrchestrator(provider)

    result = orchestrator.run(migration_plan.steps[0])

    print("STATUS:", result.status)
    print("LANGUAGE:", result.language)
    print("ASSUMPTIONS:", result.assumptions)
    print("UNRESOLVED ITEMS:", result.unresolved_items)
    print("VALIDATION REQUIREMENTS:", result.validation_requirements)
    print("\nGENERATED CODE:\n")
    print(result.code)

finally:
    os.environ.pop("DATABRICKS_TOKEN", None)
    os.environ.pop("DATABRICKS_BASE_URL", None)

In [0]:
# Actual SQL Server procedures for City Dimension migration.
# These definitions become evidence for the AI generator.

city_procedure_definitions = {
    "Integration.GetCityUpdates": r"""
    USE [WideWorldImporters]
    GO
    /****** Object:  StoredProcedure [Integration].[GetCityUpdates]    Script Date: 11-10-2026 14:40:32 ******/
    SET ANSI_NULLS ON
    GO
    SET QUOTED_IDENTIFIER ON
    GO

    ALTER PROCEDURE [Integration].[GetCityUpdates]
    @LastCutoff datetime2(7),
    @NewCutoff datetime2(7)
    WITH EXECUTE AS OWNER
    AS
    BEGIN
        SET NOCOUNT ON;
        SET XACT_ABORT ON;

        DECLARE @EndOfTime datetime2(7) = '99991231 23:59:59.9999999';
        DECLARE @InitialLoadDate date = '20130101';

        CREATE TABLE #CityChanges
        (
            [WWI City ID] int,
            City nvarchar(50),
            [State Province] nvarchar(50),
            Country nvarchar(50),
            Continent nvarchar(30),
            [Sales Territory] nvarchar(50),
            Region nvarchar(30),
            Subregion nvarchar(30),
            [Location] geography,
            [Latest Recorded Population] bigint,
            [Valid From] datetime2(7),
            [Valid To] datetime2(7) NULL
        );

        DECLARE @CountryID int;
        DECLARE @StateProvinceID int;
        DECLARE @CityID int;
        DECLARE @ValidFrom datetime2(7);

        -- first need to find any country changes that have occurred since initial load

        DECLARE CountryChangeList CURSOR FAST_FORWARD READ_ONLY
        FOR
        SELECT co.CountryID,
            co.ValidFrom
        FROM [Application].Countries_Archive AS co
        WHERE co.ValidFrom > @LastCutoff
        AND co.ValidFrom <= @NewCutoff
        AND co.ValidFrom <> @InitialLoadDate
        UNION ALL
        SELECT co.CountryID,
            co.ValidFrom
        FROM [Application].Countries AS co
        WHERE co.ValidFrom > @LastCutoff
        AND co.ValidFrom <= @NewCutoff
        AND co.ValidFrom <> @InitialLoadDate
        ORDER BY ValidFrom;

        OPEN CountryChangeList;
        FETCH NEXT FROM CountryChangeList INTO @CountryID, @ValidFrom;

        WHILE @@FETCH_STATUS = 0
        BEGIN
            INSERT #CityChanges
                ([WWI City ID], City, [State Province], Country, Continent, [Sales Territory], Region, Subregion,
                [Location], [Latest Recorded Population], [Valid From], [Valid To])
            SELECT c.CityID, c.CityName, sp.StateProvinceName, co.CountryName, co.Continent, sp.SalesTerritory, co.Region, co.Subregion,
                c.[Location], COALESCE(c.LatestRecordedPopulation, 0), @ValidFrom, NULL
            FROM [Application].Cities FOR SYSTEM_TIME AS OF @ValidFrom AS c
            INNER JOIN [Application].StateProvinces FOR SYSTEM_TIME AS OF @ValidFrom AS sp
            ON c.StateProvinceID = sp.StateProvinceID
            INNER JOIN [Application].Countries FOR SYSTEM_TIME AS OF @ValidFrom AS co
            ON sp.CountryID = co.CountryID
            WHERE co.CountryID = @CountryID;

            FETCH NEXT FROM CountryChangeList INTO @CountryID, @ValidFrom;
        END;

        CLOSE CountryChangeList;
        DEALLOCATE CountryChangeList;

        -- next need to find any stateprovince changes that have occurred since initial load

        DECLARE StateProvinceChangeList CURSOR FAST_FORWARD READ_ONLY
        FOR
        SELECT sp.StateProvinceID,
            sp.ValidFrom
        FROM [Application].StateProvinces_Archive AS sp
        WHERE sp.ValidFrom > @LastCutoff
        AND sp.ValidFrom <= @NewCutoff
        AND sp.ValidFrom <> @InitialLoadDate
        UNION ALL
        SELECT sp.StateProvinceID,
            sp.ValidFrom
        FROM [Application].StateProvinces AS sp
        WHERE sp.ValidFrom > @LastCutoff
        AND sp.ValidFrom <= @NewCutoff
        AND sp.ValidFrom <> @InitialLoadDate
        ORDER BY ValidFrom;

        OPEN StateProvinceChangeList;
        FETCH NEXT FROM StateProvinceChangeList INTO @StateProvinceID, @ValidFrom;

        WHILE @@FETCH_STATUS = 0
        BEGIN
            INSERT #CityChanges
                ([WWI City ID], City, [State Province], Country, Continent, [Sales Territory], Region, Subregion,
                [Location], [Latest Recorded Population], [Valid From], [Valid To])
            SELECT c.CityID, c.CityName, sp.StateProvinceName, co.CountryName, co.Continent, sp.SalesTerritory, co.Region, co.Subregion,
                c.[Location], COALESCE(c.LatestRecordedPopulation, 0), @ValidFrom, NULL
            FROM [Application].Cities FOR SYSTEM_TIME AS OF @ValidFrom AS c
            INNER JOIN [Application].StateProvinces FOR SYSTEM_TIME AS OF @ValidFrom AS sp
            ON c.StateProvinceID = sp.StateProvinceID
            INNER JOIN [Application].Countries FOR SYSTEM_TIME AS OF @ValidFrom AS co
            ON sp.CountryID = co.CountryID
            WHERE sp.StateProvinceID = @StateProvinceID;

            FETCH NEXT FROM StateProvinceChangeList INTO @StateProvinceID, @ValidFrom;
        END;

        CLOSE StateProvinceChangeList;
        DEALLOCATE StateProvinceChangeList;

        -- finally need to find any city changes that have occurred, including during the initial load

        DECLARE CityChangeList CURSOR FAST_FORWARD READ_ONLY
        FOR
        SELECT c.CityID,
            c.ValidFrom
        FROM [Application].Cities_Archive AS c
        WHERE c.ValidFrom > @LastCutoff
        AND c.ValidFrom <= @NewCutoff
        UNION ALL
        SELECT c.CityID,
            c.ValidFrom
        FROM [Application].Cities AS c
        WHERE c.ValidFrom > @LastCutoff
        AND c.ValidFrom <= @NewCutoff
        ORDER BY ValidFrom;

        OPEN CityChangeList;
        FETCH NEXT FROM CityChangeList INTO @CityID, @ValidFrom;

        WHILE @@FETCH_STATUS = 0
        BEGIN
            INSERT #CityChanges
                ([WWI City ID], City, [State Province], Country, Continent, [Sales Territory], Region, Subregion,
                [Location], [Latest Recorded Population], [Valid From], [Valid To])
            SELECT c.CityID, c.CityName, sp.StateProvinceName, co.CountryName, co.Continent, sp.SalesTerritory, co.Region, co.Subregion,
                c.[Location], COALESCE(c.LatestRecordedPopulation, 0), @ValidFrom, NULL
            FROM [Application].Cities FOR SYSTEM_TIME AS OF @ValidFrom AS c
            INNER JOIN [Application].StateProvinces FOR SYSTEM_TIME AS OF @ValidFrom AS sp
            ON c.StateProvinceID = sp.StateProvinceID
            INNER JOIN [Application].Countries FOR SYSTEM_TIME AS OF @ValidFrom AS co
            ON sp.CountryID = co.CountryID
            WHERE c.CityID = @CityID;

            FETCH NEXT FROM CityChangeList INTO @CityID, @ValidFrom;
        END;

        CLOSE CityChangeList;
        DEALLOCATE CityChangeList;

        -- add an index to make lookups faster

        CREATE INDEX IX_CityChanges ON #CityChanges ([WWI City ID], [Valid From]);

        -- work out the [Valid To] value by taking the [Valid From] of any row that's for the same city but later
        -- otherwise take the end of time

        UPDATE cc
        SET [Valid To] = COALESCE((SELECT MIN([Valid From]) FROM #CityChanges AS cc2
                                                            WHERE cc2.[WWI City ID] = cc.[WWI City ID]
                                                            AND cc2.[Valid From] > cc.[Valid From]), @EndOfTime)
        FROM #CityChanges AS cc;

        SELECT [WWI City ID], City, [State Province], Country, Continent, [Sales Territory],
            Region, Subregion, [Location] geography, [Latest Recorded Population], [Valid From],
            [Valid To]
        FROM #CityChanges
        ORDER BY [Valid From];

        DROP TABLE #CityChanges;

        RETURN 0;
    END;
    """,
    "Integration.MigrateStagedCityData": r"""
    USE [WideWorldImportersDW]
    GO
    /****** Object:  StoredProcedure [Integration].[MigrateStagedCityData]    Script Date: 11-10-2026 14:42:01 ******/
    SET ANSI_NULLS ON
    GO
    SET QUOTED_IDENTIFIER ON
    GO

    ALTER PROCEDURE [Integration].[MigrateStagedCityData]
    WITH EXECUTE AS OWNER
    AS
    BEGIN
        SET NOCOUNT ON;
        SET XACT_ABORT ON;

        DECLARE @EndOfTime datetime2(7) =  '99991231 23:59:59.9999999';

        BEGIN TRAN;

        DECLARE @LineageKey int = (SELECT TOP(1) [Lineage Key]
                                FROM Integration.Lineage
                                WHERE [Table Name] = N'City'
                                AND [Data Load Completed] IS NULL
                                ORDER BY [Lineage Key] DESC);

        WITH RowsToCloseOff
        AS
        (
            SELECT c.[WWI City ID], MIN(c.[Valid From]) AS [Valid From]
            FROM Integration.City_Staging AS c
            GROUP BY c.[WWI City ID]
        )
        UPDATE c
            SET c.[Valid To] = rtco.[Valid From]
        FROM Dimension.City AS c
        INNER JOIN RowsToCloseOff AS rtco
        ON c.[WWI City ID] = rtco.[WWI City ID]
        WHERE c.[Valid To] = @EndOfTime;

        INSERT Dimension.City
            ([WWI City ID], City, [State Province], Country, Continent,
            [Sales Territory], Region, Subregion, [Location],
            [Latest Recorded Population], [Valid From], [Valid To],
            [Lineage Key])
        SELECT [WWI City ID], City, [State Province], Country, Continent,
            [Sales Territory], Region, Subregion, [Location],
            [Latest Recorded Population], [Valid From], [Valid To],
            @LineageKey
        FROM Integration.City_Staging;

        UPDATE Integration.Lineage
            SET [Data Load Completed] = SYSDATETIME(),
                [Was Successful] = 1
        WHERE [Lineage Key] = @LineageKey;

        UPDATE Integration.[ETL Cutoff]
            SET [Cutoff Time] = (SELECT [Source System Cutoff Time]
                                FROM Integration.Lineage
                                WHERE [Lineage Key] = @LineageKey)
        FROM Integration.[ETL Cutoff]
        WHERE [Table Name] = N'City';

        COMMIT;

        RETURN 0;
    END;
    """,
}

for name, definition in city_procedure_definitions.items():
    if "Paste the complete" in definition:
        raise ValueError(
            f"Replace the placeholder with the real SQL for {name}"
        )

print("Procedure definitions loaded:")
for name, definition in city_procedure_definitions.items():
    print(f"  {name}: {len(definition)} characters")

In [0]:
import os

# Get the current Databricks workspace URL
workspace_url = (
    "https://" + spark.conf.get("spark.databricks.workspaceUrl")
).rstrip("/")

# Get the temporary token for this running notebook session.
# Never print or display the token.
token = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .getOrElse(None)
)

if not token:
    raise RuntimeError(
        "Databricks notebook token is unavailable. "
        "Check that this notebook is attached to a running compute."
    )

os.environ["DATABRICKS_TOKEN"] = token
os.environ["DATABRICKS_BASE_URL"] = (
    f"{workspace_url}/ai-gateway/mlflow/v1"
)

print("Databricks AI Gateway configuration is ready.")
print("Workspace URL:", workspace_url)
print("Token configured: Yes")

In [0]:
import importlib
import sys

agent_src = str(repo_root / "src")
if agent_src not in sys.path:
    sys.path.insert(0, agent_src)

import ssis_migration.agent.context_builder as context_builder_module
import ssis_migration.agent.orchestrator as orchestrator_module
import ssis_migration.agent.provider_factory as provider_factory_module

importlib.reload(context_builder_module)
importlib.reload(orchestrator_module)
importlib.reload(provider_factory_module)

# Recreate the provider and orchestrator using the reloaded classes.
provider = provider_factory_module.create_provider("databricks")
orchestrator = orchestrator_module.AIMigrationOrchestrator(provider)

# Regenerate only the first migration step.
city_step = migration_plan.steps[0]

result = orchestrator.run(
    city_step,
    procedure_definitions=city_procedure_definitions,
)

print("Step:", city_step.name)
print("Status:", result.status)
print("Language:", result.language)
print("Code length:", len(result.code or ""))
print("\nAssumptions:", result.assumptions)
print("\nUnresolved items:", result.unresolved_items)
print("\nValidation requirements:", result.validation_requirements)

In [0]:
print(result.code)

In [0]:
import importlib

import ssis_migration.agent.prompt_builder as prompt_builder_module
import ssis_migration.agent.orchestrator as orchestrator_module
import ssis_migration.agent.provider_factory as provider_factory_module

importlib.reload(prompt_builder_module)
importlib.reload(orchestrator_module)
importlib.reload(provider_factory_module)

# Recreate the provider and orchestrator with the updated prompt.
provider = provider_factory_module.create_provider("databricks")
orchestrator = orchestrator_module.AIMigrationOrchestrator(provider)

result = orchestrator.run(
    migration_plan.steps[0],
    procedure_definitions=city_procedure_definitions,
)

print("Step:", migration_plan.steps[0].name)
print("Status:", result.status)
print("Language:", result.language)
print("Code length:", len(result.code or ""))
print("\nAssumptions:", result.assumptions)
print("\nUnresolved items:", result.unresolved_items)
print("\nValidation requirements:", result.validation_requirements)

print("\n--- GENERATED CODE ---\n")
print(result.code)

In [0]:

from ssis_migration.agent.code_validator import validate_generated_code
import json

validation = validate_generated_code(
    code=result.code,
    step_name="Load City Dimension",
)

print("STATIC VALIDATION:", validation["status"])
print("=" * 70)

for issue in validation["issues"]:
    line = f" | line {issue['line']}" if issue["line"] else ""
    print(
        f"[{issue['severity']}] "
        f"{issue['code']}{line}: {issue['message']}"
    )
print("\nFull validation report:")
print(json.dumps(validation, indent=2))

if validation["status"] == "REJECTED":
    print("Migration code rejected. Do not execute or publish it.")
elif validation["status"] == "REVIEW_REQUIRED":
    print("Manual review required before accepting this migration.")
else:
    print("Static checks passed. Semantic and data validation are still required.")


In [0]:
from pathlib import Path
import inspect
import ssis_migration.agent.orchestrator as orchestrator_module

print("ORCHESTRATOR FILE:")
print(inspect.getfile(orchestrator_module))

print("\nORCHESTRATOR METHODS:")
for name, member in inspect.getmembers(
    orchestrator_module.AIMigrationOrchestrator,
    predicate=inspect.isfunction,
):
    print(name, inspect.signature(member))



In [0]:

from ssis_migration.analyzer.temporal_history import check_temporal_history

source_tables = [
    "Application.Cities",
    "Application.StateProvinces",
    "Application.Countries",
]

temporal_checks = []

for table_name in source_tables:
    try:
        df = spark.table(table_name)

        check = check_temporal_history(
            table_name=table_name,
            schema=df.schema,
            # Supply verified keys and validity columns after inspection.
            business_key_columns=[],
            valid_from_column=None,
            valid_to_column=None,
            history_confirmed=False,
        )
    except Exception as exc:
        check = {
            "table_name": table_name,
            "status": "UNRESOLVED",
            "missing_requirements": [
                f"Unable to inspect table: {type(exc).__name__}: {exc}"
            ],
        }

    temporal_checks.append(check)
    print(f"\n{table_name}: {check['status']}")
    for item in check.get("missing_requirements", []):
        print(" -", item)

print("\nColumn names:")
for table_name in source_tables:
    try:
        print(f"\n{table_name}")
        print(spark.table(table_name).columns)
    except Exception as exc:
        print(f"Cannot inspect: {exc}")


In [0]:
# ============================================================
# Discover source tables available in Databricks
# ============================================================

print("CURRENT CATALOG:", spark.sql(
    "SELECT current_catalog()"
).first()[0])

print("CURRENT SCHEMA:", spark.sql(
    "SELECT current_schema()"
).first()[0])

print("\nAVAILABLE CATALOGS")
spark.sql("SHOW CATALOGS").show(truncate=False)

print("\nAVAILABLE SCHEMAS")
spark.sql("SHOW SCHEMAS IN workspace").show(truncate=False)

In [0]:
# ============================================================
# Find City, StateProvince, and Country tables
# ============================================================

tables = spark.sql("SHOW TABLES").collect()

keywords = (
    "city",
    "stateprovince",
    "country",
    "staging",
)

matches = [
    row.asDict()
    for row in tables
    if any(
        keyword in row["tableName"].lower()
        for keyword in keywords
    )
]

if matches:
    for table in matches:
        print(table)
else:
    print(
        "No matching tables found in the current schema. "
        "Search the other catalogs and schemas."
    )

In [0]:
# READ-ONLY DISCOVERY: no catalogs, schemas, or tables are created or modified.

from datetime import datetime

print("=" * 80)
print("DATABRICKS ENVIRONMENT")
print("=" * 80)
print("Checked at:", datetime.now().isoformat(timespec="seconds"))

for label, query in [
    ("Current catalog", "SELECT current_catalog()"),
    ("Current schema", "SELECT current_schema()"),
    ("Current user", "SELECT current_user()"),
]:
    try:
        print(f"{label}: {spark.sql(query).first()[0]}")
    except Exception as exc:
        print(f"{label}: ERROR - {str(exc)[:300]}")

try:
    print("\nAVAILABLE CATALOGS")
    spark.sql("SHOW CATALOGS").show(truncate=False)
except Exception as exc:
    print("Catalog discovery failed:", str(exc)[:500])

In [0]:
# READ-ONLY: scans metadata; does not read table contents or modify objects.

catalogs = [
    row["catalog"]
    for row in spark.sql("SHOW CATALOGS").collect()
]

all_schemas = []
all_tables = []
all_views = []
scan_errors = []

for catalog in catalogs:
    try:
        schemas = [
            row[0]
            for row in spark.sql(
                f"SHOW SCHEMAS IN `{catalog}`"
            ).collect()
        ]
    except Exception as exc:
        scan_errors.append(
            ("SCHEMAS", catalog, str(exc)[:250])
        )
        continue

    for schema in schemas:
        all_schemas.append((catalog, schema))

        try:
            rows = spark.sql(
                f"SHOW TABLES IN `{catalog}`.`{schema}`"
            ).collect()

            for row in rows:
                all_tables.append(
                    (catalog, schema, row["tableName"])
                )
        except Exception as exc:
            scan_errors.append(
                ("TABLES", f"{catalog}.{schema}", str(exc)[:250])
            )

        try:
            rows = spark.sql(
                f"SHOW VIEWS IN `{catalog}`.`{schema}`"
            ).collect()

            for row in rows:
                all_views.append(
                    (catalog, schema, row["viewName"])
                )
        except Exception:
            # Some schemas or runtimes may not support this operation.
            pass

print("=" * 80)
print(f"CATALOGS: {len(catalogs)}")
print(f"SCHEMAS DISCOVERED: {len(all_schemas)}")
print(f"TABLES DISCOVERED: {len(all_tables)}")
print(f"VIEWS DISCOVERED: {len(all_views)}")
print("=" * 80)

print("\nALL ACCESSIBLE SCHEMAS")
for catalog, schema in sorted(all_schemas):
    print(f"{catalog}.{schema}")

print("\nALL DISCOVERED TABLES")
for catalog, schema, table in sorted(all_tables):
    print(f"{catalog}.{schema}.{table}")

print("\nALL DISCOVERED VIEWS")
for catalog, schema, view in sorted(all_views):
    print(f"{catalog}.{schema}.{view}")

print("\nMETADATA SCAN ERRORS")
if scan_errors:
    for kind, location, error in scan_errors:
        print(f"{kind} | {location} | {error}")
else:
    print("None recorded")

In [0]:
keywords = (
    "wideworldimporters",
    "city",
    "stateprovince",
    "country",
    "customer",
    "stockitem",
    "staging",
    "lineage",
    "cutoff",
    "dimension",
    "integration",
    "fact",
)

print("=" * 80)
print("RELEVANT REGISTERED OBJECTS")
print("=" * 80)

matches = []

for catalog, schema, table in all_tables:
    searchable = f"{catalog}.{schema}.{table}".lower()

    if any(keyword in searchable for keyword in keywords):
        matches.append(
            ("TABLE", f"{catalog}.{schema}.{table}")
        )

for catalog, schema, view in all_views:
    searchable = f"{catalog}.{schema}.{view}".lower()

    if any(keyword in searchable for keyword in keywords):
        matches.append(
            ("VIEW", f"{catalog}.{schema}.{view}")
        )

if matches:
    for kind, name in sorted(set(matches)):
        print(f"{kind}: {name}")
else:
    print("No relevant tables or views found.")

In [0]:
column_keywords = (
    "city",
    "stateprovince",
    "country",
    "staging",
    "lineage",
    "cutoff",
)

candidate_tables = [
    (catalog, schema, table)
    for catalog, schema, table in all_tables
    if any(keyword in table.lower() for keyword in column_keywords)
]

print("=" * 80)
print("RELEVANT TABLE COLUMNS")
print("=" * 80)

if not candidate_tables:
    print("No candidate tables to inspect.")
else:
    for catalog, schema, table in sorted(candidate_tables):
        full_name = f"`{catalog}`.`{schema}`.`{table}`"
        print(f"\n--- {catalog}.{schema}.{table} ---")

        try:
            spark.sql(f"DESCRIBE TABLE {full_name}").show(
                200,
                truncate=False,
            )
        except Exception as exc:
            print("Could not describe table:", str(exc)[:500])

In [0]:
from pathlib import Path

candidate_roots = [
    Path("/Workspace/Users/sridsaravanan.ext@deloitte.com/ssis-migration-agent"),
    Path("/Workspace/Shared"),
]

file_extensions = {".dtsx", ".sql", ".json", ".yaml", ".yml", ".csv"}

print("=" * 80)
print("PROJECT FILE DISCOVERY")
print("=" * 80)

for root in candidate_roots:
    print(f"\nROOT: {root}")

    if not root.exists():
        print("Not accessible as a local filesystem path.")
        continue

    count = 0

    try:
        for path in root.rglob("*"):
            if path.is_file() and path.suffix.lower() in file_extensions:
                print(path)
                count += 1

                if count >= 200:
                    print("Output capped at 200 files for this root.")
                    break

        if count == 0:
            print("No matching files found under this root.")
    except Exception as exc:
        print("Could not scan root:", str(exc)[:500])

In [0]:
import json
import re
from pathlib import Path

PROJECT = Path(
    "/Workspace/Users/sridsaravanan.ext@deloitte.com/"
    "ssis-migration-agent/examples/wwi"
)

with open(PROJECT / "database_metadata_wwi_dw.json", encoding="utf-8") as f:
    dw_metadata = json.load(f)

# Explicit target mapping. These are proposed names, not deployed objects.
TARGET_CATALOG = "dev_catalog"

SCHEMA_MAP = {
    "Dimension": "dimension",
    "Fact": "fact",
    "Integration": "integration",
    "Application": "application",
}

# Explicit exceptions where SQL Server names contain spaces.
TABLE_NAME_MAP = {
    ("Integration", "ETL Cutoff"): "etl_cutoff",
}

TYPE_MAP = {
    "int": "INT",
    "bigint": "BIGINT",
    "smallint": "SMALLINT",
    "tinyint": "TINYINT",
    "bit": "BOOLEAN",
    "date": "DATE",
    "datetime": "TIMESTAMP",
    "datetime2": "TIMESTAMP",
    "smalldatetime": "TIMESTAMP",
    "float": "DOUBLE",
    "real": "FLOAT",
    "money": "DECIMAL(19,4)",
    "smallmoney": "DECIMAL(10,4)",
    "uniqueidentifier": "STRING",
    "nvarchar": "STRING",
    "varchar": "STRING",
    "nchar": "STRING",
    "char": "STRING",
    "text": "STRING",
    "ntext": "STRING",
    "xml": "STRING",
    "binary": "BINARY",
    "varbinary": "BINARY",
    "image": "BINARY",
    "sysname": "STRING",
}

def normalize_name(name):
    # Lowercase, replace unsupported separators with underscores.
    result = re.sub(r"[^a-zA-Z0-9_]", "_", name.strip())
    result = re.sub(r"_+", "_", result).strip("_").lower()

    if not result:
        raise ValueError(f"Cannot normalize identifier: {name!r}")

    return result

def map_type(column):
    source_type = (source_type or "").strip().lower()
    source_type = column["data_type"].lower()

    if source_type in ("decimal", "numeric"):
        precision = column.get("precision") or 18
        scale = column.get("scale") or 0
        if precision > 38:
            return None, f"DECIMAL precision {precision} exceeds Spark limit"
        if scale > precision:
            return None, f"Scale {scale} exceeds precision {precision}"
        return f"DECIMAL({precision},{scale})", None

    if source_type in ("nvarchar", "varchar", "nchar", "char"):
        return "STRING", None

    if source_type in TYPE_MAP:
        return TYPE_MAP[source_type], None

    # These require an intentional migration policy.
    if source_type == "geography":
        return None, "Define spatial representation before migration"
    if source_type == "geometry":
        return None, "Define spatial representation before migration"
    if source_type == "time":
        return None, "Define SQL Server TIME representation"
    if source_type == "sql_variant":
        return None, "Define heterogeneous-value representation"

    return None, f"No type mapping for SQL Server type {source_type}"

ddl_statements = []
review_items = []

for table in dw_metadata.get("tables", []):
    source_schema = table["schema"]
    source_table = table["name"]

    if source_schema not in SCHEMA_MAP:
        review_items.append({
            "object": f"{source_schema}.{source_table}",
            "issue": "No target schema mapping configured",
        })
        continue

    target_schema = SCHEMA_MAP[source_schema]
    target_table = TABLE_NAME_MAP.get(
        (source_schema, source_table),
        normalize_name(source_table),
    )

    columns_sql = []

    for column in sorted(
        table.get("columns", []),
        key=lambda c: c.get("ordinal", 0),
    ):
        target_type, issue = map_type(column)

        if issue:
            review_items.append({
                "object": f"{source_schema}.{source_table}",
                "column": column["name"],
                "source_type": column["data_type"],
                "issue": issue,
            })
            # Do not silently guess a replacement type.
            continue

        nullable = "" if column.get("nullable", True) else " NOT NULL"
        column_name = normalize_name(column["name"])

        columns_sql.append(
            f"  `{column_name}` {target_type}{nullable}"
        )

    if len(columns_sql) != len(table.get("columns", [])):
        review_items.append({
            "object": f"{source_schema}.{source_table}",
            "issue": "DDL omitted one or more columns; table is incomplete",
        })
        continue

    full_name = (
        f"`{TARGET_CATALOG}`.`{target_schema}`.`{target_table}`"
    )

    ddl_statements.append(
        f"CREATE TABLE IF NOT EXISTS {full_name} (\n"
        + ",\n".join(columns_sql)
        + "\n)\nUSING DELTA;"
    )

ddl_path = PROJECT / "generated_dw_ddl_preview.sql"
review_path = PROJECT / "generated_dw_ddl_review.json"

ddl_path.write_text(
    "-- REVIEW ONLY. DO NOT EXECUTE UNTIL APPROVED.\n\n"
    + "\n\n".join(ddl_statements)
    + "\n",
    encoding="utf-8",
)

review_path.write_text(
    json.dumps(review_items, indent=2),
    encoding="utf-8",
)

print("DDL preview:", ddl_path)
print("Review report:", review_path)
print("Destination tables included:", len(ddl_statements))
print("Unresolved review items:", len(review_items))

print("\nFirst 100 lines of DDL preview:")
print("\n".join(ddl_path.read_text(encoding="utf-8").splitlines()[:100]))

print("\nFirst 30 review items:")
print(json.dumps(review_items[:30], indent=2))

In [0]:
import json
from pathlib import Path

base = Path(
    "/Workspace/Users/sridsaravanan.ext@deloitte.com/"
    "ssis-migration-agent/examples/wwi"
)

report_path = base / "generated_dw_ddl_review.json"
ddl_path = base / "generated_dw_ddl_preview.sql"

report = json.loads(report_path.read_text())
ddl = ddl_path.read_text()

print("Review report type:", type(report).__name__)

if isinstance(report, dict):
    print("Top-level keys:", list(report.keys()))
    items = report.get("unresolved_items", report.get("review_items", []))
else:
    items = report

print("\nIncomplete-table review entries:")
for item in items:
    if "incomplete" in item.get("issue", "").lower():
        print(item)

print("\nGenerated DDL sections for affected tables:")
for table in [
    "dimension.city",
    "integration.city_staging",
    "integration.etl cutoff",
    "integration.lineage",
]:
    print(f"\n--- {table} ---")
    lines = ddl.splitlines()
    start = next(
        (
            i for i, line in enumerate(lines)
            if table.lower() in line.lower()
            and "CREATE TABLE" in line.upper()
        ),
        None,
    )
    if start is None:
        print("CREATE TABLE statement not found")
        continue

    end = next(
        (
            i for i in range(start + 1, len(lines))
            if lines[i].strip().endswith(";")
        ),
        len(lines),
    )
    print("\n".join(lines[start:end + 1]))

In [0]:
from pathlib import Path

repo = Path(
    "/Workspace/Users/sridsaravanan.ext@deloitte.com/"
    "ssis-migration-agent"
)

print("Repository exists:", repo.exists())
print("Repository path:", repo)
print("Top-level entries:")

if repo.exists():
    for item in sorted(repo.iterdir()):
        print("DIR " if item.is_dir() else "FILE", item.name)

In [0]:
from pathlib import Path

repo = Path(
    "/Workspace/Users/sridsaravanan.ext@deloitte.com/"
    "ssis-migration-agent"
)

# Search source code, scripts, notebooks and tests.
search_dirs = [
    "src",
    "scripts",
    "notebooks",
    "tests",
]

terms = [
    "CREATE TABLE",
    "incomplete",
    "review",
    "ddl",
    "column_mapping",
    "generate",
]

for dirname in search_dirs:
    root = repo / dirname

    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        if path.suffix.lower() not in {
            ".py", ".sql", ".ipynb", ".json", ".md"
        }:
            continue

        if any(
            part in {".git", "__pycache__", ".venv", "venv",
                     "node_modules"}
            for part in path.parts
        ):
            continue

        try:
            content = path.read_text(encoding="utf-8")
        except (OSError, UnicodeDecodeError):
            continue

        found = [
            (i, line.strip()[:200])
            for i, line in enumerate(content.splitlines(), 1)
            if any(term.lower() in line.lower() for term in terms)
        ]

        if found:
            print(f"\nFILE: {path.relative_to(repo)}")
            for line_no, line in found[:12]:
                print(f"  {line_no}: {line}")

In [0]:
import json
from pathlib import Path

repo = Path(
    "/Workspace/Users/sridsaravanan.ext@deloitte.com/"
    "ssis-migration-agent"
)

metadata_path = repo / "examples/wwi/database_metadata_wwi_dw.json"

with metadata_path.open(encoding="utf-8") as f:
    metadata = json.load(f)

print("TOP-LEVEL KEYS:")
print(list(metadata.keys()))

tables = metadata.get("tables", [])

print("\nTABLE COUNT:", len(tables))

if tables:
    first_table = tables[0]

    print("\nFIRST TABLE STRUCTURE:")
    print(json.dumps(first_table, indent=2)[:5000])

    print("\nCOLUMN STRUCTURE:")
    columns = first_table.get("columns", [])
    if columns:
        print(json.dumps(columns[0], indent=2))
    else:
        print("No columns found under the expected 'columns' key.")

In [0]:
from pathlib import Path

path = Path(
    "/Workspace/Users/sridsaravanan.ext@deloitte.com/"
    "ssis-migration-agent/src/ssis_migration/ddl_generator.py"
)

print("Exists:", path.exists())

if path.exists():
    print("File size:", path.stat().st_size, "bytes")
    print("First 15 lines:")
    print("\n".join(path.read_text().splitlines()[:15]))

In [0]:
import json
import sys
from pathlib import Path

repo = Path(
    "/Workspace/Users/sridsaravanan.ext@deloitte.com/"
    "ssis-migration-agent"
)

sys.path.insert(0, str(repo / "src"))

from ssis_migration.ddl_generator import generate_ddl

metadata_path = repo / "examples/wwi/database_metadata_wwi_dw.json"

with metadata_path.open(encoding="utf-8") as f:
    metadata = json.load(f)

ddl, report = generate_ddl(
    metadata=metadata,
    catalog="dev_catalog",
)

ddl_path = repo / "examples/wwi/generated_dw_ddl_preview.sql"
report_path = repo / "examples/wwi/generated_dw_ddl_review.json"

ddl_path.write_text(ddl, encoding="utf-8")
report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

print("Status:", report["status"])
print("Metadata tables:", report["table_count"])
print("Generated tables:", report["generated_table_count"])
print("Review findings:", len(report["review_items"]))

print("\nFirst five review findings:")
for item in report["review_items"][:5]:
    print(item)

print("\nDDL preview:")
print(ddl[:3000])

In [0]:
from pathlib import Path

repo = Path(
    "/Workspace/Users/sridsaravanan.ext@deloitte.com/ssis-migration-agent"
)

for relative_path in [
    "src/ssis_migration/ddl_generator.py",
    "pyproject.toml",
]:
    path = repo / relative_path
    print(f"\n{'=' * 70}\n{relative_path}\n{'=' * 70}")
    print(path.read_text(encoding="utf-8"))

In [0]:
import json
import sys
from pathlib import Path

repo = Path(
    "/Workspace/Users/sridsaravanan.ext@deloitte.com/ssis-migration-agent"
)
sys.path.insert(0, str(repo / "src"))

from ssis_migration.ddl_generator import generate_ddl

metadata_path = repo / "examples/wwi/database_metadata_wwi_dw.json"

with metadata_path.open(encoding="utf-8") as f:
    metadata = json.load(f)

ddl, report = generate_ddl(
    metadata=metadata,
    catalog="dev_catalog",
    column_mapping=True,
)

(repo / "examples/wwi/generated_dw_ddl_preview.sql").write_text(
    ddl, encoding="utf-8"
)
(repo / "examples/wwi/generated_dw_ddl_review.json").write_text(
    json.dumps(report, indent=2), encoding="utf-8"
)

print("Status:", report["status"])
print("Generated tables:", report["generated_table_count"])
print("Review findings:", len(report["review_items"]))
print("\nFirst DDL statement:\n")
print(ddl.split(";")[0] + ";")

In [0]:
%%writefile /Workspace/Users/sridsaravanan.ext@deloitte.com/ssis-migration-agent/tests/test_provisioning.py
import pytest

from ssis_migration.provisioning import (
    plan_schema_provisioning,
    required_schemas,
)


def test_derives_schemas_from_metadata():
    metadata = {
        "tables": [
            {"schema": "Dimension", "name": "City"},
            {"schema": "Fact", "name": "Sale"},
            {"schema": "dimension", "name": "Customer"},
        ]
    }

    assert required_schemas(metadata) == ["dimension", "fact"]


def test_plans_only_missing_schemas():
    metadata = {
        "tables": [
            {"schema": "Dimension", "name": "City"},
            {"schema": "Fact", "name": "Sale"},
            {"schema": "Integration", "name": "Lineage"},
        ]
    }

    plan = plan_schema_provisioning(
        metadata=metadata,
        catalog="dev_catalog",
        existing_schemas=["default", "dimension"],
    )

    assert plan["required_schemas"] == [
        "dimension", "fact", "integration"
    ]
    assert plan["missing_schemas"] == ["fact", "integration"]
    assert len(plan["statements"]) == 2
    assert plan["execution_enabled"] is False


def test_existing_schemas_need_no_changes():
    metadata = {
        "tables": [
            {"schema": "Dimension", "name": "City"},
        ]
    }

    plan = plan_schema_provisioning(
        metadata,
        "dev_catalog",
        ["dimension"],
    )

    assert plan["missing_schemas"] == []
    assert plan["statements"] == []
    assert plan["status"] == "READY"


def test_invalid_metadata_is_rejected():
    with pytest.raises(ValueError):
        required_schemas({"tables": "not-a-list"})

In [0]:
import json
import sys
from pathlib import Path

# Repository location
repo = Path(
    "/Workspace/Users/sridsaravanan.ext@deloitte.com/ssis-migration-agent"
)

# Make project modules importable
sys.path.insert(0, str(repo / "src"))

# Import the generator
from ssis_migration.ddl_generator import generate_ddl

# Load destination metadata
metadata_path = repo / "examples/wwi/database_metadata_wwi_dw.json"

with metadata_path.open(encoding="utf-8") as f:
    metadata = json.load(f)

# Generate DDL; do not execute it
ddl, report = generate_ddl(
    metadata=metadata,
    catalog="dev_catalog",
    column_mapping=True,
)

# Save the generated SQL and review report
ddl_path = repo / "examples/wwi/generated_dw_ddl_preview.sql"
report_path = repo / "examples/wwi/generated_dw_ddl_review.json"

ddl_path.write_text(ddl, encoding="utf-8")
report_path.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("DDL GENERATION COMPLETE")
print("Status:", report["status"])
print("Generated tables:", report["generated_table_count"])
print("Review findings:", len(report["review_items"]))
print("DDL file:", ddl_path)
print("Review report:", report_path)

for item in report["review_items"]:
    print(item)


In [0]:
from pathlib import Path
import re

repo = Path(
    "/Workspace/Users/sridsaravanan.ext@deloitte.com/ssis-migration-agent"
)

ddl_path = repo / "examples/wwi/generated_dw_ddl_preview.sql"
ddl = ddl_path.read_text(encoding="utf-8")

# Count CREATE TABLE statements in the preview.
statements = [
    statement.strip()
    for statement in ddl.split(";")
    if statement.strip()
]

create_table_statements = [
    statement + ";"
    for statement in statements
    if re.match(r"(?is)^CREATE\s+TABLE\s+IF\s+NOT\s+EXISTS\b", statement)
]

print("DDL file exists:", ddl_path.is_file())
print("CREATE TABLE statements:", len(create_table_statements))
print("GEOGRAPHY(4326) occurrences:", ddl.upper().count("GEOGRAPHY(4326)"))
print("Has column mapping property:", "delta.columnMapping.mode" in ddl)

print("\nFirst generated statement:\n")
print(create_table_statements[0] if create_table_statements else "NONE")

In [0]:
import json
import sys
from pathlib import Path

repo = Path(
    "/Workspace/Users/sridsaravanan.ext@deloitte.com/ssis-migration-agent"
)
sys.path.insert(0, str(repo / "src"))

from ssis_migration.migration_runner import run_structure_migration

result = run_structure_migration(
    spark=spark,
    metadata_path=str(
        repo / "examples/wwi/database_metadata_wwi_dw.json"
    ),
    catalog="dev_catalog",
    output_dir=str(repo / "examples/wwi"),
    execute=False,
    column_mapping=True,
)

print("STATUS:", result["status"])
print("SCHEMAS:", result["schema_provisioning"]["status"])
print("TABLES:", result["table_creation"]["planned_statement_count"])
print("TABLE EXECUTED:", result["table_creation"]["executed"])
print("DATA:", result["data_migration"]["status"])
print("REVIEW FINDINGS:", len(result["table_creation"]["review_findings"]))
print(json.dumps(result["table_creation"]["results"][:2], indent=2))

In [0]:

import sys
from pathlib import Path

repo = Path(
    "/Workspace/Users/sridsaravanan.ext@deloitte.com/"
    "ssis-migration-agent"
)
sys.path.insert(0, str(repo / "src"))

from ssis_migration.migrate import run_migration

report = run_migration(
    source=str(repo / "examples/wwi/DailyETLMain.dtsx"),
    output=str(repo / "generated_migrations"),
    catalog="dev_catalog",
)

print("STATUS:", report["status"])
print("PACKAGES:", report["package_count"])
print("ANALYZED:", report["analyzed_package_count"])
print("FAILED:", report["failed_package_count"])

for package in report["packages"]:
    print("\nPACKAGE:", package["name"])
    print("STATUS:", package["status"])
    if package.get("error"):
        print("ERROR:", package["error"])
    print("ARTIFACTS:", package.get("artifacts", {}))


In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

TARGET_DIR = (
    "/Users/sridsaravanan.ext@deloitte.com/"
    "ssis-migration-agent/DailyETLMain_ai_baseline"
)
TARGET_NOTEBOOK = TARGET_DIR + "/DailyETLMain_Migration"

print("=== WORKSPACE RESOURCE CHECK ===")

for label, path in [
    ("Target directory", TARGET_DIR),
    ("Migration notebook", TARGET_NOTEBOOK),
]:
    try:
        obj = w.workspace.get_status(path)
        print(f"{label}: EXISTS")
        print("  Path:", obj.path)
        print("  Type:", obj.object_type)
    except Exception as e:
        print(f"{label}: NOT FOUND OR NOT ACCESSIBLE")
        print(" ", str(e)[:500])